# Agentic Search with Shell tool

This notebook walks you through an agentic search example with a shell tool that lets the agent run commands in the terminal to navigate a local filesystem and use `grep` to search files. It builds on the first implementation by adding `jina-grep-cli` as a semantic search alternative to `grep`.

This approach lets the agent pull context stored in a local filesystem into its context window to answer **user queries containing keywords** and handle **conceptual questions**.

<img src="../img/Agentic search with shell tool.png" width="1000" alt="Agentic search with shell tool" />

## Preparation

Make sure you have completed the setup instructions according to the README.md.

Load environment variables from `.env` file, such as API keys.

In [3]:
import os                                                                                                                                                                                                          
from dotenv import load_dotenv

load_dotenv(override=True)

LITELLM_API_KEY=os.getenv("AZURE_API_KEY")
LITELLM_API_BASE=os.getenv("AZURE_API_BASE")

ES_URL=os.getenv("ELASTICSEARCH_URL")
ES_USERNAME=os.getenv("ELASTICSEARCH_USERNAME")
ES_PASSWORD=os.getenv("ELASTICSEARCH_PASSWORD")

## Set up agent with shell tool (and `grep`)

### Configure the LLM

Set up your LLM with the right parameters for your use case:
Below, we're using OpenAI's models through LiteLLM but you can switch it out with any model capable of tool use.

In [4]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=LITELLM_API_BASE,
    api_key=LITELLM_API_KEY,
    deployment_name="gpt-5.4",
    temperature=0.5,
    api_version="2024-12-01-preview",
)

### Define the system prompt

The system prompt defines your agent’s role and behavior. 

We also include information about what out-of-context knowledge source it has. In this case it has access to a **local filesystem** with the conference data.

In [5]:
from IPython.display import display, Markdown

with open("system_prompt_fs.md", 'r') as f:
    SYSTEM_PROMPT = f.read()

display(Markdown(SYSTEM_PROMPT))

You are a search agent tasked with answering questions about the AI Engineer Europe 2026 Conference.

You have access to different context retrieval tools to help you answer user queries.

Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
If the retrieved context does not contain relevant information to answer the query, say that you don't know. 

## Local filesystem (`session_data`)

The conference sessions are available under under `../data/session_data/`. One file per session.

File structure:
```
../
data/
 └── session_data/
  ├── workshop/
  │ ├── <title>.txt
  │ └── ... 
  ├── ...
  └── expo_session/
```

File Sample:
```
# <Title>

- **Day:** <Date of the session>
- **Time:** <Time slot of the session>
- **Room:** <Room where the session takes place>
- **Type:** <One of 'keynote', 'workshop', 'talk', 'track_keynote', 'lightning', 'expo_session'>
- **Speakers:** <Name(s) of the speaker(s)>

<Description>
```

### Define built-in shell tool

Let's define a built-in [Shell tool](https://docs.langchain.com/oss/python/integrations/tools/bash) the agent can use to execute commands in a terminal. A common use case for this is letting the LLM interact with your local file system.

<div class="alert alert-block alert-warning">
Giving agents access to the shell is powerful but also risky when used outside of a sandboxed environment because it doesn't  have any safeguards by default. <b>Use at your own risk.</b>

Note: Shell tool does not work with Windows OS.
</div>

In [6]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

# Test the shell tool
print(shell_tool.run({"commands": ["echo 'Hello World!'"]}))

Executing command:
 ["echo 'Hello World!'"]
Hello World!



/workspaces/workshop-agentic-search/venv/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


### Create the agent

Now assemble your agent with all the components including the built-in shell tool.

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[shell_tool],
)

## Run the agent

In [10]:
input_message = {
    "role": "user",
    "content": ("Are there any sessions about GEPA?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Are there any sessions about GEPA?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_5ExP296uMVeDg501g9lOaFc7)
 Call ID: call_5ExP296uMVeDg501g9lOaFc7
  Args:
    commands: grep -Ril "GEPA" ../data/session_data || true
Executing command:
 ['grep -Ril "GEPA" ../data/session_data || true']
================================= Tool Message =================================
Name: terminal

../data/session_data/workshop/Playground in Prod - Optimising Agents in Production Environments.txt



/workspaces/workshop-agentic-search/venv/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================

Yes — there is at least one session mentioning GEPA:

- **Playground in Prod - Optimising Agents in Production Environments**
  - **Type:** Workshop

If you want, I can also give you the **day, time, room, and speakers** for that session.


In [9]:
input_message = {
    "role": "user",
    "content": ("Which sessions discuss handling regulatory constraints?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions discuss handling regulatory constraints?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_oaEPAGsZpmHycqa2b77eLkh6)
 Call ID: call_oaEPAGsZpmHycqa2b77eLkh6
  Args:
    commands: grep -Ril "regulator\|regulatory\|compliance\|constraint" ../data/session_data | sed 's#^../data/session_data/##'
Executing command:
 ['grep -Ril "regulator\\|regulatory\\|compliance\\|constraint" ../data/session_data | sed \'s#^../data/session_data/##\'']
================================= Tool Message =================================
Name: terminal

talk/Lessons from Scaling GitHub's Remote MCP Server.txt
talk/Connecting the Dots with Context Graphs.txt
talk/Scaling the Next Paradigm of Heterogeneous Intelligence.txt
talk/Engineering AI Systems Under Sovereignty Constraints.txt



/workspaces/workshop-agentic-search/venv/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_a2yr6ggVjVRzGfuuLSoc6gFW)
 Call ID: call_a2yr6ggVjVRzGfuuLSoc6gFW
  Args:
    commands: for f in "../data/session_data/talk/Lessons from Scaling GitHub's Remote MCP Server.txt" "../data/session_data/talk/Connecting the Dots with Context Graphs.txt" "../data/session_data/talk/Scaling the Next Paradigm of Heterogeneous Intelligence.txt" "../data/session_data/talk/Engineering AI Systems Under Sovereignty Constraints.txt"; do echo "---FILE:$f---"; sed -n '1,120p' "$f"; done
Executing command:
 ['for f in "../data/session_data/talk/Lessons from Scaling GitHub\'s Remote MCP Server.txt" "../data/session_data/talk/Connecting the Dots with Context Graphs.txt" "../data/session_data/talk/Scaling the Next Paradigm of Heterogeneous Intelligence.txt" "../data/session_data/talk/Engineering AI Systems Under Sovereignty Constraints.txt"; do echo "---FILE:$f---"; sed -n \'1,120p\' "$f"; done']
=

/workspaces/workshop-agentic-search/venv/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================

The session most directly about handling regulatory constraints is:

- **Engineering AI Systems Under Sovereignty Constraints**  
  - **Day:** April 9  
  - **Time:** 12:40–1:00pm  
  - **Room:** St. James  
  - **Speaker:** Bilge Yücel  

This talk explicitly focuses on **regulatory and jurisdictional constraints** in AI system design, including:
- data localization
- auditability
- model transparency
- jurisdictional control
- compliance-oriented architecture decisions

There is also one session that touches related policy/constraint themes more indirectly:

- **Connecting the Dots with Context Graphs**  
  - mentions **constraints** and **policies** as part of context graphs, but it is not specifically about regulatory constraints.

I didn’t find other sessions clearly centered on regulatory constraints.


## Set up agent with shell tool (and semantic `grep` alternative)

`grep` is based on regular expressions and is great for pattern matching. However, it has the limitation that it's not great at semantic search.

There are many alternatives to `grep` available that enable semantic search over files, for example [`jina-grep-cli`](https://github.com/jina-ai/jina-grep-cli).

### 1. Install the `jina-grep` CLI

Install the `jina-grep` CLI in your virtual environment:

```bash
git clone https://github.com/jina-ai/jina-grep-cli.git && cd jina-grep-cli
uv pip install -e .
```

### 2. Adjust the agent's system prompt

Next, you need to tell your agent about the new CLI it can use in its system prompt. Make sure to also add some explanations on how to use the `jina-grep` CLI.

In [11]:
with open("system_prompt_fs_jina_grep.md", 'r') as f:
    SYSTEM_PROMPT = f.read()

print(SYSTEM_PROMPT)

You are a search agent tasked with answering questions about the AI Engineer Europe 2026 Conference.

You have access to different context retrieval tools to help you answer user queries.

Before answering a question decide whether or not you need to retrieve additional context to answer the question correctly.
If the retrieved context does not contain relevant information to answer the query, say that you don't know. 

## Local filesystem (`session_data`)

The conference sessions are available under under `../data/session_data/`. One file per session.

File structure:
```
../
data/
 └── session_data/
  ├── workshop/
  │ ├── <title>.txt
  │ └── ... 
  ├── ...
  └── expo_session/
```

File Sample:
```
# <Title>

- **Day:** <Date of the session>
- **Time:** <Time slot of the session>
- **Room:** <Room where the session takes place>
- **Type:** <One of 'keynote', 'workshop', 'talk', 'track_keynote', 'lightning', 'expo_session'>
- **Speakers:** <Name(s) of the speaker(s)>

<Description>
``

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[shell_tool],
)

In [13]:
input_message = {
    "role": "user",
    "content": ("Which sessions discuss handling regulatory constraints?"),
}

for step in agent.stream(
    {"messages": [input_message]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which sessions discuss handling regulatory constraints?
================================== Ai Message ==================================
Tool Calls:
  terminal (call_wwJzYbpE51CIMMvJ0Z7HtcmK)
 Call ID: call_wwJzYbpE51CIMMvJ0Z7HtcmK
  Args:
    commands: jina-grep -r --top-k 10 --threshold 0.35 "handling regulatory constraints compliance governance legal requirements regulated industries" ../data/session_data
Executing command:
 ['jina-grep -r --top-k 10 --threshold 0.35 "handling regulatory constraints compliance governance legal requirements regulated industries" ../data/session_data']
================================= Tool Message =================================
Name: terminal

/bin/sh: 1: jina-grep: not found



/workspaces/workshop-agentic-search/venv/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_ZU2FPMc59M8pkjvfDiPuKMTX)
 Call ID: call_ZU2FPMc59M8pkjvfDiPuKMTX
  Args:
    commands: grep -RilE "regulat|compliance|governance|legal|regulated|constraints" ../data/session_data | head -n 20
Executing command:
 ['grep -RilE "regulat|compliance|governance|legal|regulated|constraints" ../data/session_data | head -n 20']
================================= Tool Message =================================
Name: terminal

../data/session_data/talk/Lessons from Scaling GitHub's Remote MCP Server.txt
../data/session_data/talk/Connecting the Dots with Context Graphs.txt
../data/session_data/talk/Scaling the Next Paradigm of Heterogeneous Intelligence.txt
../data/session_data/talk/What Does Multi-Modal and Coding Have to Do with Legal.txt
../data/session_data/talk/Engineering AI Systems Under Sovereignty Constraints.txt



/workspaces/workshop-agentic-search/venv/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  terminal (call_1XClRlJdXKXq1T1t3z10jAhb)
 Call ID: call_1XClRlJdXKXq1T1t3z10jAhb
  Args:
    commands: ["sed -n '1,220p' ../data/session_data/talk/Engineering\\ AI\\ Systems\\ Under\\ Sovereignty\\ Constraints.txt", "sed -n '1,220p' ../data/session_data/talk/What\\ Does\\ Multi-Modal\\ and\\ Coding\\ Have\\ to\\ Do\\ with\\ Legal.txt", "sed -n '1,220p' ../data/session_data/talk/Connecting\\ the\\ Dots\\ with\\ Context\\ Graphs.txt", "sed -n '1,220p' ../data/session_data/talk/Lessons\\ from\\ Scaling\\ GitHub\\'s\\ Remote\\ MCP\\ Server.txt", "sed -n '1,220p' ../data/session_data/talk/Scaling\\ the\\ Next\\ Paradigm\\ of\\ Heterogeneous\\ Intelligence.txt"]
Executing command:
 ["sed -n '1,220p' ../data/session_data/talk/Engineering\\ AI\\ Systems\\ Under\\ Sovereignty\\ Constraints.txt", "sed -n '1,220p' ../data/session_data/talk/What\\ Does\\ Multi-Modal\\ and\\ Coding\\ Have\\ to\\ Do\\ with

/workspaces/workshop-agentic-search/venv/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


================================== Ai Message ==================================

The session most directly about handling regulatory constraints is:

- **Engineering AI Systems Under Sovereignty Constraints** — **April 9, 12:40–1:00pm, St. James**  
  This talk explicitly focuses on **regulatory and jurisdictional constraints**, including **data localization, auditability, model transparency, jurisdictional control, and governance by design**.

Other sessions that touch adjacent themes:

- **What Does Multi-Modal and Coding Have to Do with Legal?** — **April 9, 12:40–1:00pm, Abbey**  
  Relevant if you mean **legal-domain applications**, though it’s more about applying multimodal AI and coding in legal workflows than about regulatory compliance itself.

- **Connecting the Dots with Context Graphs** — **April 9, 2:30–2:50pm, St. James**  
  Mentions **constraints, policies, provenance, explainability, and trustworthy enterprise AI**, so it may be useful from a governance perspective.

